In [1]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE
from neuralforecast.losses.numpy import mae, rmse
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

all_tickers = df['Ticker'].unique().tolist()

ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()

df_filtered = df[df['Ticker'].isin(valid_tickers)]

# Подготовка данных
def prepare_data(df, tickers):
    data_list = []
    for ticker in tickers:
        ticker_data = df[df['Ticker'] == ticker].sort_values('date').copy()
        data_list.append(pd.DataFrame({
            'unique_id': ticker,
            'ds': ticker_data['date'],
            'y': ticker_data['Close']
        }))
    return pd.concat(data_list, ignore_index=True)

data = prepare_data(df_filtered, valid_tickers)


# Разделение: до 2023 - train, 2024 - test
TRAIN_END = '2023-12-31'
TEST_START = '2024-01-01'

train_data = data[data['ds'] <= TRAIN_END]
test_data = data[data['ds'] >= TEST_START]


def train_and_evaluate(train_df, test_df, horizon, input_size=60, max_steps=200):
    
    # Для длинных горизонтов используем val_size=0
    if horizon > 90:
        val_size = 0
        print(f"  (использую val_size=0 для горизонта {horizon} дней)")
    else:
        val_size = min(30, 500)  # достаточно большой
    
    model = NeuralForecast(
        models=[
            NHITS(
                h=horizon,
                input_size=input_size,
                max_steps=max_steps,
                batch_size=128,  
                learning_rate=1e-3,
                
                stack_types=["identity", "identity", "identity"],
                n_blocks=[1, 1, 1],
                mlp_units=[[256, 256], [256, 256], [256, 256]],
                n_pool_kernel_size=[2, 2, 1],
                n_freq_downsample=[4, 2, 1],
                
                scaler_type='robust',
                random_seed=42,
            )
        ],
        freq='D'
    )
    
    model.fit(df=train_df, val_size=val_size)

    forecast = model.predict()
    
    test_forecast = forecast[forecast['ds'] >= TEST_START]
    
    all_rmse = []
    all_mae = []
    ticker_metrics = {}
    
    for ticker in train_df['unique_id'].unique():
        ticker_test = test_df[test_df['unique_id'] == ticker].sort_values('ds')
        ticker_forecast = test_forecast[test_forecast['unique_id'] == ticker].sort_values('ds')
        
        if len(ticker_test) > 0 and len(ticker_forecast) > 0:
            min_len = min(len(ticker_test), len(ticker_forecast))
            if min_len > 0:
                y_true = ticker_test['y'].iloc[:min_len].values
                y_pred = ticker_forecast['NHITS'].iloc[:min_len].values
                
                ticker_rmse = rmse(y_true, y_pred)
                ticker_mae = mae(y_true, y_pred)
                
                if not np.isnan(ticker_rmse) and not np.isnan(ticker_mae):
                    all_rmse.append(ticker_rmse)
                    all_mae.append(ticker_mae)
                    ticker_metrics[ticker] = {'RMSE': ticker_rmse, 'MAE': ticker_mae}
    
    if len(all_rmse) > 0:
        return np.mean(all_rmse), np.mean(all_mae), ticker_metrics
    else:
        return np.nan, np.nan, {}



# Определяем горизонты
horizons = {
    'На месяц (30 дней)': 30,
    'На год по неделям (52 недели)': 364,
    'На год по месяцам (12 месяцев)': 365
}

results = {}
all_ticker_metrics = {}

for name, horizon in horizons.items():    
    # Выбираем параметры в зависимости от горизонта
    if horizon <= 30:
        input_size = 60
        max_steps = 200
    elif horizon <= 90:
        input_size = 120
        max_steps = 300
    else:
        input_size = 180
        max_steps = 500
    
    print(f"Использую input_size={input_size}, max_steps={max_steps}")
    
    try:
        avg_rmse, avg_mae, ticker_metrics = train_and_evaluate(
            train_data, test_data, 
            horizon=horizon, 
            input_size=input_size,
            max_steps=max_steps
        )
        
        results[name] = {
            'horizon_days': horizon,
            'avg_rmse': avg_rmse,
            'avg_mae': avg_mae
        }
        
        all_ticker_metrics[name] = ticker_metrics
        
    except Exception as e:
        print(f"✗ Ошибка: {e}")
        results[name] = {
            'horizon_days': horizon,
            'avg_rmse': np.nan,
            'avg_mae': np.nan
        }
        all_ticker_metrics[name] = {}


summary = pd.DataFrame([
    {
        'Горизонт': name,
        'Дней': info['horizon_days'],
        'RMSE ($)': info['avg_rmse'] if not np.isnan(info['avg_rmse']) else 'N/A',
        'MAE ($)': info['avg_mae'] if not np.isnan(info['avg_mae']) else 'N/A'
    }
    for name, info in results.items()
])

print("СВОДНАЯ ТАБЛИЦА МЕТРИК (ВСЕ ТИКЕРЫ):")
print(summary.to_string(index=False))

summary.to_csv('forecast_metrics_all_tickers_1.csv', index=False)


for name, ticker_metrics in all_ticker_metrics.items():
    if ticker_metrics:
        print(f"{name} (горизонт {horizons[name]} дней):")
        
        df_metrics = pd.DataFrame(ticker_metrics).T
        
        print(f"  Количество тикеров: {len(df_metrics)}")
        print(f"  Средний RMSE: ${df_metrics['RMSE'].mean():.4f}")
        print(f"  Медианный RMSE: ${df_metrics['RMSE'].median():.4f}")
        print(f"  Средний MAE: ${df_metrics['MAE'].mean():.4f}")
        print(f"  Медианный MAE: ${df_metrics['MAE'].median():.4f}")
        
        df_metrics.to_csv(f'ticker_metrics_{name.replace(" ", "_")}_1.csv')
        print(f"\n  ✓ Детальные метрики сохранены в 'ticker_metrics_{name.replace(" ", "_")}_1.csv'")


Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Использую input_size=60, max_steps=200


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  683 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 683 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 683 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Output()

`Trainer.fit` stopped: `max_steps=500` reached.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

СВОДНАЯ ТАБЛИЦА МЕТРИК (ВСЕ ТИКЕРЫ):
                      Горизонт  Дней  RMSE ($)   MAE ($)
            На месяц (30 дней)    30 10.353539  8.891856
 На год по неделям (52 недели)   364 41.834263 34.741817
На год по месяцам (12 месяцев)   365 41.741305 34.658109
На месяц (30 дней) (горизонт 30 дней):
  Количество тикеров: 199
  Средний RMSE: $10.3535
  Медианный RMSE: $5.8779
  Средний MAE: $8.8919
  Медианный MAE: $4.8248

  ✓ Детальные метрики сохранены в 'ticker_metrics_На_месяц_(30_дней)_1.csv'
На год по неделям (52 недели) (горизонт 364 дней):
  Количество тикеров: 199
  Средний RMSE: $41.8343
  Медианный RMSE: $24.2451
  Средний MAE: $34.7418
  Медианный MAE: $19.3495

  ✓ Детальные метрики сохранены в 'ticker_metrics_На_год_по_неделям_(52_недели)_1.csv'
На год по месяцам (12 месяцев) (горизонт 365 дней):
  Количество тикеров: 199
  Средний RMSE: $41.7413
  Медианный RMSE: $23.8819
  Средний MAE: $34.6581
  Медианный MAE: $19.3308

  ✓ Детальные метрики сохранены в 'ticker_metr

In [169]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE
from neuralforecast.losses.numpy import mae, rmse
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['Ticker', 'date'])


# Лаги
for lag in [1, 5, 10, 20]:
    df[f'close_lag_{lag}'] = df.groupby('Ticker')['Close'].shift(lag)

# Скользящие средние
for window in [5, 10, 20, 50]:
    df[f'ma_{window}'] = df.groupby('Ticker')['Close'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )

# Доходности
df['return_1d'] = df.groupby('Ticker')['Close'].pct_change(1)
df['return_5d'] = df.groupby('Ticker')['Close'].pct_change(5)
df['return_10d'] = df.groupby('Ticker')['Close'].pct_change(10)
df['log_return_1d'] = df.groupby('Ticker')['Close'].transform(
    lambda x: np.log(x / x.shift(1))
)

# Волатильность
for window in [5, 10, 20]:
    df[f'volatility_{window}'] = df.groupby('Ticker')['Close'].transform(
        lambda x: x.rolling(window, min_periods=1).std()
    )

# RSI
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['rsi_14'] = df.groupby('Ticker')['Close'].transform(lambda x: compute_rsi(x, 14))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line

macd_data = []
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker].sort_values('date')
    macd_line, signal_line = compute_macd(ticker_data['Close'])
    
    ticker_df = pd.DataFrame({
        'Ticker': ticker,
        'date': ticker_data['date'].values,
        'macd_line': macd_line.values,
        'macd_signal': signal_line.values
    })
    macd_data.append(ticker_df)

macd_df = pd.concat(macd_data, ignore_index=True)
df = df.merge(macd_df, on=['Ticker', 'date'], how='left')

df['hl_ratio'] = df['High'] / df['Low']
df['co_ratio'] = df['Close'] / df['Open']
df['range_pct'] = (df['High'] - df['Low']) / df['Open']

df['volume_relative'] = df.groupby('Ticker')['Volume'].transform(
    lambda x: x / x.rolling(20, min_periods=1).mean()
)


exog_features = [
    'return_1d', 'return_5d', 'return_10d', 'log_return_1d',
    'volatility_5', 'volatility_10', 'volatility_20',
    'rsi_14',
    'macd_line', 'macd_signal',
    'hl_ratio', 'co_ratio', 'range_pct',
    'volume_relative'
]

# Проверяем наличие признаков
available_exog = [col for col in exog_features if col in df.columns]

# Фильтрация тикеров
ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()
df_filtered = df[df['Ticker'].isin(valid_tickers)]

# Подготовка данных
def prepare_data_with_hist_exog(df, tickers, exog_cols):

    data_list = []
    for ticker in tickers:
        ticker_data = df[df['Ticker'] == ticker].sort_values('date').copy()
        
        ticker_df = pd.DataFrame({
            'unique_id': ticker,
            'ds': ticker_data['date'],
            'y': ticker_data['Close']
        })
        
        for col in exog_cols:
            if col in ticker_data.columns:
                ticker_df[col] = ticker_data[col].values
        
        data_list.append(ticker_df)
    
    return pd.concat(data_list, ignore_index=True)

data_with_exog = prepare_data_with_hist_exog(df_filtered, valid_tickers, available_exog)

# Удаляем NaN
initial_len = len(data_with_exog)
data_with_exog = data_with_exog.dropna()


TRAIN_END = '2023-12-31'
TEST_START = '2024-01-01'

train_data = data_with_exog[data_with_exog['ds'] <= TRAIN_END]
test_data = data_with_exog[data_with_exog['ds'] >= TEST_START]



def train_with_hist_exog(train_df, test_df, horizon, exog_cols, input_size=60, max_steps=200):
    
    val_size = 0 if horizon > 90 else 30
    
    model = NeuralForecast(
        models=[
            NHITS(
                h=horizon,
                input_size=input_size,
                max_steps=max_steps,
                batch_size=64,
                learning_rate=1e-3,
                
                hist_exog_list=exog_cols,
                
                stack_types=["identity", "identity", "identity"],
                n_blocks=[1, 1, 1],
                mlp_units=[[128, 128], [128, 128], [128, 128]],
                n_pool_kernel_size=[2, 2, 1],
                n_freq_downsample=[4, 2, 1],
                
                scaler_type='robust',
                random_seed=42,
            )
        ],
        freq='D'
    )
    
    model.fit(df=train_df, val_size=val_size)
    
    forecast = model.predict()
    
    test_forecast = forecast[forecast['ds'] >= TEST_START]
    
    all_rmse, all_mae = [], []
    
    for ticker in train_df['unique_id'].unique():
        ticker_test = test_df[test_df['unique_id'] == ticker].sort_values('ds')
        ticker_forecast = test_forecast[test_forecast['unique_id'] == ticker].sort_values('ds')
        
        if len(ticker_test) > 0 and len(ticker_forecast) > 0:
            min_len = min(len(ticker_test), len(ticker_forecast))
            if min_len > 0:
                y_true = ticker_test['y'].iloc[:min_len].values
                y_pred = ticker_forecast['NHITS'].iloc[:min_len].values
                all_rmse.append(rmse(y_true, y_pred))
                all_mae.append(mae(y_true, y_pred))
    
    if all_rmse:
        return np.mean(all_rmse), np.mean(all_mae)
    return np.nan, np.nan


rmse_with_exog, mae_with_exog = train_with_hist_exog(
    train_data, test_data, 
    horizon=30, 
    exog_cols=available_exog,
    input_size=60,
    max_steps=200
)

print(f"РЕЗУЛЬТАТЫ С ИСТОРИЧЕСКИМИ ПРИЗНАКАМИ:")
print(f"  RMSE: ${rmse_with_exog:.4f}")
print(f"  MAE:  ${mae_with_exog:.4f}")


rmse_without = 10.26
mae_without = 8.82

print(f"СРАВНЕНИЕ (прогноз на месяц):")
print("-" * 60)
print(f"{'Метрика':<15} {'Без признаков':<20} {'С признаками':<20} {'Изменение':<15}")
print(f"{'RMSE':<15} ${rmse_without:<19.2f} ${rmse_with_exog:<19.2f} {((rmse_without - rmse_with_exog)/rmse_without*100):+.1f}%")
print(f"{'MAE':<15} ${mae_without:<19.2f} ${mae_with_exog:<19.2f} {((mae_without - mae_with_exog)/mae_without*100):+.1f}%")


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  409 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 409 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 409 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_steps=200` reached.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

РЕЗУЛЬТАТЫ С ИСТОРИЧЕСКИМИ ПРИЗНАКАМИ:
  RMSE: $10.2832
  MAE:  $8.8373
СРАВНЕНИЕ (прогноз на месяц):
------------------------------------------------------------
Метрика         Без признаков        С признаками         Изменение      
RMSE            $10.26               $10.28               -0.2%
MAE             $8.82                $8.84                -0.2%


In [170]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE
from neuralforecast.losses.numpy import mae, rmse
import warnings
warnings.filterwarnings('ignore')


df_stocks = pd.read_csv('prices_all.csv')
df_stocks['date'] = pd.to_datetime(df_stocks['date'])
df_stocks = df_stocks.sort_values(['Ticker', 'date'])



df_indexes = pd.read_csv('Main_indexs.csv')

# Обработка дат
if 'Date' in df_indexes.columns:
    df_indexes['date'] = pd.to_datetime(df_indexes['Date'], utc=True)
    df_indexes['date'] = df_indexes['date'].dt.tz_localize(None)
    df_indexes.drop(columns=['Date'], inplace=True)
else:
    df_indexes['date'] = pd.to_datetime(df_indexes['date'])

if 'Close' in df_indexes.columns:
    df_indexes.rename(columns={'Close': 'close'}, inplace=True)

# Приводим к date (без времени) для правильного merge
df_stocks['date'] = df_stocks['date'].dt.date
df_indexes['date'] = df_indexes['date'].dt.date



# Создаем pivot таблицу
index_pivot = df_indexes.pivot_table(
    index='date', 
    columns='Ticker', 
    values='close'
)
index_pivot.columns = [f'idx_{col}' for col in index_pivot.columns]
index_pivot = index_pivot.reset_index()


# Создаем DataFrame со всеми датами из акций
all_dates = pd.DataFrame({
    'date': sorted(df_stocks['date'].unique())
})


# Объединяем
full_index_df = all_dates.merge(index_pivot, on='date', how='left')


# Заполняем пропуски
index_cols = [col for col in full_index_df.columns if col.startswith('idx_')]

for col in index_cols:
    # Заполняем вперед
    full_index_df[col] = full_index_df[col].fillna(method='ffill')
    # Заполняем назад для начальных пропусков
    full_index_df[col] = full_index_df[col].fillna(method='bfill')
    # Если все еще есть пропуски, заполняем 0
    full_index_df[col] = full_index_df[col].fillna(0)

df_merged = df_stocks.merge(full_index_df, on='date', how='left')




# Используем ВСЕ индексы
available_indices = index_cols.copy()


ticker_counts = df_merged.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()


df_filtered = df_merged[df_merged['Ticker'].isin(valid_tickers)]



def prepare_data_with_exog(df, tickers, exog_cols):
    data_list = []
    for ticker in tickers:
        ticker_data = df[df['Ticker'] == ticker].sort_values('date').copy()
        
        # Преобразуем date обратно в datetime для модели
        ticker_data['date'] = pd.to_datetime(ticker_data['date'])
        
        ticker_df = pd.DataFrame({
            'unique_id': ticker,
            'ds': ticker_data['date'],
            'y': ticker_data['Close']
        })
        
        for col in exog_cols:
            if col in ticker_data.columns:
                ticker_df[col] = ticker_data[col].values
        
        data_list.append(ticker_df)
    
    return pd.concat(data_list, ignore_index=True)

data_with_exog = prepare_data_with_exog(df_filtered, valid_tickers, available_indices)

# Удаляем NaN
initial_len = len(data_with_exog)
data_with_exog = data_with_exog.dropna()



TRAIN_END = '2023-12-31'
TEST_START = '2024-01-01'

train_data = data_with_exog[data_with_exog['ds'] <= TRAIN_END]
test_data = data_with_exog[data_with_exog['ds'] >= TEST_START]



def train_with_exog(train_df, test_df, horizon, exog_cols, input_size=60, max_steps=200):
    
    val_size = 0 if horizon > 90 else 30
    
    model = NeuralForecast(
        models=[
            NHITS(
                h=horizon,
                input_size=input_size,
                max_steps=max_steps,
                batch_size=64,
                learning_rate=1e-3,
                
                # Исторические экзогенные признаки (все индексы)
                hist_exog_list=exog_cols,
                
                stack_types=["identity", "identity", "identity"],
                n_blocks=[1, 1, 1],
                mlp_units=[[256, 256], [256, 256], [256, 256]],
                n_pool_kernel_size=[2, 2, 1],
                n_freq_downsample=[4, 2, 1],
                
                scaler_type='robust',
                random_seed=42,
            )
        ],
        freq='D'
    )
    
    print(f"  Обучение на {train_df['unique_id'].nunique()} тикерах...")
    model.fit(df=train_df, val_size=val_size)
    
    print(f"  Генерация прогнозов...")
    forecast = model.predict()
    
    test_forecast = forecast[forecast['ds'] >= TEST_START]
    
    all_rmse, all_mae = [], []
    
    for ticker in train_df['unique_id'].unique():
        ticker_test = test_df[test_df['unique_id'] == ticker].sort_values('ds')
        ticker_forecast = test_forecast[test_forecast['unique_id'] == ticker].sort_values('ds')
        
        if len(ticker_test) > 0 and len(ticker_forecast) > 0:
            min_len = min(len(ticker_test), len(ticker_forecast))
            if min_len > 0:
                y_true = ticker_test['y'].iloc[:min_len].values
                y_pred = ticker_forecast['NHITS'].iloc[:min_len].values
                all_rmse.append(rmse(y_true, y_pred))
                all_mae.append(mae(y_true, y_pred))
    
    if all_rmse:
        return np.mean(all_rmse), np.mean(all_mae)
    return np.nan, np.nan


rmse_with_all_idx, mae_with_all_idx = train_with_exog(
    train_data, test_data, 
    horizon=30, 
    exog_cols=available_indices,
    input_size=60,
    max_steps=200
)

print(f"\nРЕЗУЛЬТАТЫ СО ВСЕМИ РЫНОЧНЫМИ ИНДЕКСАМИ:")
print(f"  RMSE: ${rmse_with_all_idx:.4f}")
print(f"  MAE:  ${mae_with_all_idx:.4f}")




results = {
    'Базовая модель': {'rmse': 10.26, 'mae': 8.82},
    'С техническими признаками': {'rmse': 10.28, 'mae': 8.84},
    'Со всеми индексами': {'rmse': rmse_with_all_idx, 'mae': mae_with_all_idx}
}


baseline_rmse = results['Базовая модель']['rmse']

for name, metrics in results.items():
    rmse_val = metrics['rmse']
    mae_val = metrics['mae']
    if name != 'Базовая модель':
        change = ((baseline_rmse - rmse_val) / baseline_rmse) * 100
        change_symbol = '+' if change > 0 else ''
        print(f"{name:<25} ${rmse_val:<14.2f} ${mae_val:<14.2f} {change_symbol}{change:.1f}%")
    else:
        print(f"{name:<25} ${rmse_val:<14.2f} ${mae_val:<14.2f} baseline")




Seed set to 42


  Обучение на 199 тикерах...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  1.3 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

`Trainer.fit` stopped: `max_steps=200` reached.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

  Генерация прогнозов...



РЕЗУЛЬТАТЫ СО ВСЕМИ РЫНОЧНЫМИ ИНДЕКСАМИ:
  RMSE: $11.0023
  MAE:  $9.3133
Базовая модель            $10.26          $8.82           baseline
С техническими признаками $10.28          $8.84           -0.2%
Со всеми индексами        $11.00          $9.31           -7.2%


In [ ]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE
from neuralforecast.losses.numpy import mae, rmse
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])


ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()

df_filtered = df[df['Ticker'].isin(valid_tickers)]

def prepare_data(df, tickers):
    data_list = []
    for ticker in tickers:
        ticker_data = df[df['Ticker'] == ticker].sort_values('date').copy()
        data_list.append(pd.DataFrame({
            'unique_id': ticker,
            'ds': ticker_data['date'],
            'y': ticker_data['Close']
        }))
    return pd.concat(data_list, ignore_index=True)

data = prepare_data(df_filtered, valid_tickers)

TRAIN_END = '2024-12-31'
TEST_START = '2025-01-01'
TEST_END = '2025-12-31'

train_data = data[data['ds'] <= TRAIN_END]
test_data = data[(data['ds'] >= TEST_START) & (data['ds'] <= TEST_END)]

def train_and_evaluate(train_df, test_df, horizon, input_size=60, max_steps=200):
    val_size = 0 if horizon > 90 else 30
    
    model = NeuralForecast(
        models=[
            NHITS(
                h=horizon,
                input_size=input_size,
                max_steps=max_steps,
                batch_size=128,
                learning_rate=1e-3,
                stack_types=["identity", "identity", "identity"],
                n_blocks=[1, 1, 1],
                mlp_units=[[256, 256], [256, 256], [256, 256]],
                n_pool_kernel_size=[2, 2, 1],
                n_freq_downsample=[4, 2, 1],
                scaler_type='robust',
                random_seed=42,
            )
        ],
        freq='D'
    )
    
    model.fit(df=train_df, val_size=val_size)
    forecast = model.predict()
    
    test_forecast = forecast[(forecast['ds'] >= TEST_START) & (forecast['ds'] <= TEST_END)]
    
    all_rmse, all_mae = [], []
    
    for ticker in train_df['unique_id'].unique():
        ticker_test = test_df[test_df['unique_id'] == ticker].sort_values('ds')
        ticker_forecast = test_forecast[test_forecast['unique_id'] == ticker].sort_values('ds')
        
        if len(ticker_test) > 0 and len(ticker_forecast) > 0:
            min_len = min(len(ticker_test), len(ticker_forecast))
            if min_len > 0:
                y_true = ticker_test['y'].iloc[:min_len].values
                y_pred = ticker_forecast['NHITS'].iloc[:min_len].values
                all_rmse.append(rmse(y_true, y_pred))
                all_mae.append(mae(y_true, y_pred))
    
    if len(all_rmse) > 0:
        return model, np.mean(all_rmse), np.mean(all_mae)
    return model, np.nan, np.nan

horizons = {
    'На месяц (30 дней)': 30,
    'На год по неделям (52 недели)': 364,
    'На год по месяцам (12 месяцев)': 365
}

results = {}
models = {}

for name, horizon in horizons.items():
    
    if horizon <= 30:
        input_size, max_steps = 60, 200
    elif horizon <= 90:
        input_size, max_steps = 120, 300
    else:
        input_size, max_steps = 180, 500
    
    model, avg_rmse, avg_mae = train_and_evaluate(
        train_data, test_data, 
        horizon=horizon, 
        input_size=input_size,
        max_steps=max_steps
    )
    
    results[name] = {'rmse': avg_rmse, 'mae': avg_mae}
    models[name] = model


Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  683 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 683 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 683 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_steps=200` reached.


...

Output()

`Trainer.fit` stopped: `max_steps=500` reached.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

МЕТРИКИ НА ТЕСТОВОМ ПЕРИОДЕ (2025 год):
                      Горизонт RMSE ($) MAE ($)
            На месяц (30 дней)    14.96   12.55
 На год по неделям (52 недели)    39.60   33.12
На год по месяцам (12 месяцев)    39.50   33.04


In [7]:
df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

# Фильтрация тикеров
ticker_counts = df.groupby('Ticker').size()
valid_tickers = ticker_counts[ticker_counts >= 500].index.tolist()
df_filtered = df[df['Ticker'].isin(valid_tickers)]

# Подготовка данных
def prepare_data(df, tickers):
    data_list = []
    for ticker in tickers:
        ticker_data = df[df['Ticker'] == ticker].sort_values('date').copy()
        data_list.append(pd.DataFrame({
            'unique_id': ticker,
            'ds': ticker_data['date'],
            'y': ticker_data['Close']
        }))
    return pd.concat(data_list, ignore_index=True)

data = prepare_data(df_filtered, valid_tickers)



last_date = data['ds'].max()
start_forecast = last_date + pd.Timedelta(days=1)
end_may = pd.Timestamp('2026-05-31')
forecast_dates_may = pd.date_range(start=start_forecast, end=end_may, freq='D')

if len(forecast_dates_may) > 0:    

    final_model = NeuralForecast(
        models=[NHITS(
            h=len(forecast_dates_may), 
            input_size=60, 
            max_steps=200, 
            batch_size=128, 
            scaler_type='robust'
        )],
        freq='D'
    )
    final_model.fit(df=data, val_size=0)
    
    # Прогноз
    forecast_may = final_model.predict()
    
    # Фильтруем только нужные даты
    forecast_may = forecast_may[forecast_may['ds'].isin(forecast_dates_may)]
    
    if len(forecast_may) > 0:
        forecast_may.to_csv('forecast_may_2026.csv', index=False)


forecast_dates_monthly = pd.date_range(start='2026-05-01', end='2026-12-01', freq='MS')

available_dates = [d for d in forecast_dates_monthly if d > last_date]

if len(available_dates) > 0:
    print(f"Месяцев для прогноза: {len(available_dates)}")
    print(f"Даты: {[d.date() for d in available_dates]}")
    
    # Для прогноза на месяцы используем horizon = количество оставшихся месяцев
    # Но NHITS работает с днями, поэтому переводим в дни
    # Прогнозируем до конца года
    end_of_year = pd.Timestamp('2026-12-31')
    days_to_end = (end_of_year - last_date).days
    
    if days_to_end > 0:
        
        final_model_yearly = NeuralForecast(
            models=[NHITS(
                h=days_to_end, 
                input_size=180, 
                max_steps=500, 
                batch_size=128, 
                scaler_type='robust'
            )],
            freq='D'
        )
        final_model_yearly.fit(df=data, val_size=0)
        
        # Прогноз на все дни до конца года
        forecast_all = final_model_yearly.predict()
        
        # Выбираем только первые числа месяцев
        forecast_monthly = forecast_all[forecast_all['ds'].isin(available_dates)]
        
        if len(forecast_monthly) > 0:
            forecast_monthly.to_csv('forecast_monthly_2026.csv', index=False)




Seed set to 1
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  2.7 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.7 M                                                                                                
Total estimated model params size (MB): 10                                                                         
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_steps=200` reached.


...

Trainable params: 3.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.2 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_steps=500` reached.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()